# Can an LLM be trusted with numbers? Python computes, the LLM explains

**Author:** Bilel Charfi

A small reproducible experiment on sales data. We compare two ways to build an "AI data analyst":

| Approach | Who computes the numbers? | Who writes the text? |
|---|---|---|
| **A — LLM only** | the LLM, from the raw monthly table | the LLM |
| **B — Hybrid** | Python + Pandas | the LLM, from pre-computed metrics, then checked by code |

Because the data is **synthetic with a known ground truth**, every number the LLM writes can be checked.

**What we measure**
1. A common Pandas pitfall: sorting month names alphabetically.
2. Approach A: how often the LLM gets the total and the growth wrong.
3. Approach B: how often the verification layer has to block the LLM's report.
4. Latency of each approach.

**How to run**
- Kaggle: *Settings → Accelerator → GPU T4*, and *Internet → On*. Then *Run all*.
- Google Colab or a local machine also work.
- Sections 1 to 3 run without any LLM. Sections 4 to 6 need Ollama (installed automatically in section 1).
- At the end, all results are saved in the `outputs/` folder.

## 0. Configuration

In [ ]:
# ---- Change these values if needed ----
MODEL = "qwen3.5:4b"      # Ollama model tag
N_SCENARIOS = 20          # number of synthetic companies (more = more reliable, but slower)
N_MONTHS = 12             # months of history per company
SEED = 42                 # random seed (reproducibility)
INSTALL_OLLAMA = True     # set False if Ollama is already installed and running
TOLERANCE_PCT = 0.1       # tolerance (percentage points) when checking a growth value
TOLERANCE_REL = 0.005     # relative tolerance (0.5 %) when checking a total

import os, sys, re, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "|", platform.platform())

## 1. Install and start Ollama (optional)

This cell installs Ollama on Linux (Kaggle, Colab), starts the server and downloads the model.
If anything fails, the notebook continues: sections 2 and 3 do not need an LLM.

In [ ]:
import subprocess, shutil, urllib.request

def ollama_running():
    try:
        urllib.request.urlopen("http://localhost:11434", timeout=2)
        return True
    except Exception:
        return False

LLM_AVAILABLE = False
try:
    if INSTALL_OLLAMA and shutil.which("ollama") is None and platform.system() == "Linux":
        # zstd is needed by recent versions of the install script
        subprocess.run("(command -v zstd || (apt-get -qq update && apt-get -qq install -y zstd)) > /dev/null 2>&1",
                       shell=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    if not ollama_running() and shutil.which("ollama"):
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(30):
            if ollama_running():
                break
            time.sleep(1)
    if ollama_running():
        subprocess.run(["ollama", "pull", MODEL], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama"], check=True)
        LLM_AVAILABLE = True
except Exception as e:
    print("Ollama setup failed:", e)

print("LLM available:", LLM_AVAILABLE)

## 2. Synthetic sales data with a known ground truth

Each scenario is one company with `N_MONTHS` months of sales and profit.
The last month goes **up** for about half of the companies and **down** for the others,
so a model cannot score well by always saying "sales increased".

In [ ]:
rng = np.random.default_rng(SEED)
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def make_scenario(i):
    base = rng.uniform(20_000, 200_000)
    trend = rng.uniform(-0.02, 0.03)
    sales = [base]
    for _ in range(N_MONTHS - 2):
        sales.append(sales[-1] * (1 + trend + rng.normal(0, 0.05)))
    last_change = rng.uniform(0.02, 0.20) * (1 if i % 2 == 0 else -1)   # alternate up / down
    sales.append(sales[-1] * (1 + last_change))
    sales = np.round(sales, 2)
    margin = rng.uniform(0.05, 0.25)
    profit = np.round(sales * (margin + rng.normal(0, 0.01, N_MONTHS)), 2)
    months = pd.period_range("2025-01", periods=N_MONTHS, freq="M")
    return pd.DataFrame({
        "scenario": i,
        "month": months.astype(str),            # e.g. 2025-03
        "month_name": [MONTH_NAMES[p.month - 1] for p in months],   # e.g. Mar
        "sales": sales,
        "profit": profit,
    })

data = pd.concat([make_scenario(i) for i in range(N_SCENARIOS)], ignore_index=True)
data.to_csv(f"{OUT}/synthetic_sales.csv", index=False)
data.head(13)

### Ground truth, computed by Python

In [ ]:
def compute_metrics(df):
    df = df.copy()
    df["month"] = pd.to_datetime(df["month"])
    monthly = df.groupby(df["month"].dt.to_period("M"))["sales"].sum().sort_index()
    total_sales = float(df["sales"].sum())
    total_profit = float(df["profit"].sum())
    return {
        "total_sales": round(total_sales, 2),
        "total_profit": round(total_profit, 2),
        "profit_margin_percent": round(total_profit / total_sales * 100, 2),
        "last_month_sales": round(float(monthly.iloc[-1]), 2),
        "previous_month_sales": round(float(monthly.iloc[-2]), 2),
        "monthly_growth_percent": round(float(monthly.pct_change().iloc[-1] * 100), 2),
    }

truth = {i: compute_metrics(g) for i, g in data.groupby("scenario")}
truth_df = pd.DataFrame(truth).T
truth_df["direction"] = np.where(truth_df["monthly_growth_percent"] < 0, "down", "up")
truth_df.to_csv(f"{OUT}/ground_truth.csv")
print(truth_df["direction"].value_counts().to_string())
truth_df.head()

## 3. Experiment 1 — The month-sorting pitfall (no LLM)

Many tutorials group by a month **name** (`Jan`, `Feb`, ...) and call `sort_index()`.
The sort is then alphabetical (`Apr, Aug, Dec, Feb, ...`), so "the last month" is not the last month.
With 12 months, the alphabetical order puts `Sep` last and `Oct` just before it: the code compares September with October instead of December with November.
How often does this change the result?

In [ ]:
rows = []
for i, g in data.groupby("scenario"):
    naive = g.groupby("month_name")["sales"].sum().sort_index()     # alphabetical!
    naive_growth = (naive.iloc[-1] / naive.iloc[-2] - 1) * 100
    true_growth = truth[i]["monthly_growth_percent"]
    rows.append({
        "scenario": i,
        "naive_compares": f"{naive.index[-1]} vs {naive.index[-2]}",
        "naive_growth_%": round(naive_growth, 2),
        "true_growth_%": true_growth,
        "wrong_value": abs(naive_growth - true_growth) > TOLERANCE_PCT,
        "wrong_direction": np.sign(naive_growth) != np.sign(true_growth),
    })
pitfall = pd.DataFrame(rows)
pitfall.to_csv(f"{OUT}/exp1_sort_pitfall.csv", index=False)

print(f"Wrong growth value     : {pitfall['wrong_value'].mean():.0%} of scenarios")
print(f"Wrong direction (up/down): {pitfall['wrong_direction'].mean():.0%} of scenarios")
pitfall.head(10)

## 4. LLM helpers

In [ ]:
def ask_llm(messages, json_mode=False):
    """Call the local model. Returns (text, seconds)."""
    from ollama import chat
    kwargs = dict(model=MODEL, messages=messages, options={"temperature": 0})
    if json_mode:
        kwargs["format"] = "json"
    t0 = time.time()
    try:
        r = chat(think=False, **kwargs)     # disable "thinking" for speed, when supported
    except Exception:
        r = chat(**kwargs)
    return r.message.content, time.time() - t0

def parse_number(x):
    try:
        return float(str(x).replace(",", "").replace("%", "").strip())
    except Exception:
        return np.nan

## 5. Experiment 2 — Approach A: the LLM computes from the raw table

The model receives the monthly table and must return the total sales,
the last-month growth and the direction, as JSON.

In [ ]:
SYSTEM_A = ("You are a data analyst. Compute the requested values from the table. "
            "Answer only with JSON: {\"total_sales\": number, "
            "\"monthly_growth_percent\": number, \"direction\": \"up\" or \"down\"}. "
            "monthly_growth_percent is the change of the last month versus the previous month.")

results_a = []
if LLM_AVAILABLE:
    for i, g in data.groupby("scenario"):
        table = g[["month", "sales", "profit"]].to_csv(index=False)
        text, sec = ask_llm([{"role": "system", "content": SYSTEM_A},
                             {"role": "user", "content": f"Monthly data:\n{table}"}], json_mode=True)
        try:
            ans = json.loads(text)
        except Exception:
            ans = {}
        t = truth[i]
        llm_total = parse_number(ans.get("total_sales"))
        llm_growth = parse_number(ans.get("monthly_growth_percent"))
        llm_dir = str(ans.get("direction", "")).lower().strip()
        results_a.append({
            "scenario": i, "seconds": round(sec, 2), "valid_json": bool(ans),
            "llm_total": llm_total, "true_total": t["total_sales"],
            "total_rel_error_%": round(abs(llm_total - t["total_sales"]) / t["total_sales"] * 100, 3),
            "llm_growth": llm_growth, "true_growth": t["monthly_growth_percent"],
            "growth_abs_error_pts": round(abs(llm_growth - t["monthly_growth_percent"]), 3),
            "llm_direction": llm_dir, "true_direction": "down" if t["monthly_growth_percent"] < 0 else "up",
        })
        print(f"scenario {i}: {sec:.1f}s")
    res_a = pd.DataFrame(results_a)
    res_a["total_ok"] = res_a["total_rel_error_%"] <= TOLERANCE_REL * 100
    res_a["growth_ok"] = res_a["growth_abs_error_pts"] <= TOLERANCE_PCT
    res_a["direction_ok"] = res_a["llm_direction"] == res_a["true_direction"]
    res_a.to_csv(f"{OUT}/exp2_llm_computes.csv", index=False)
    display(res_a.head(10))
else:
    print("Skipped: no LLM available.")

## 6. Experiment 3 — Approach B: Python computes, the LLM explains, code verifies

The model receives only the pre-computed metrics and writes a short report.
A verifier then checks every percentage in the text and the announced direction.
It also flags sentences that suggest a **cause** (e.g. "due to"), because the data contains no cause at all.

In [ ]:
SYSTEM_B = ("You are a data analyst writing for managers. Explain the supplied metrics in 3 short sentences. "
            "Never invent numbers or causes.")

UP_WORDS = ["increase", "increased", "grew", "rose", "growth of", "up by"]
DOWN_WORDS = ["decrease", "decreased", "decline", "declined", "dropped", "fell", "down by"]
CAUSE_WORDS = ["due to", "because", "caused by", "driven by", "as a result of", "owing to", "thanks to"]

def verify_report(text, metrics, tol=TOLERANCE_PCT):
    problems = []
    allowed = [abs(metrics["monthly_growth_percent"]), abs(metrics["profit_margin_percent"])]
    for p in re.findall(r"[-−]?\d+(?:[.,]\d+)?\s?%", text):
        v = abs(float(p.replace("%", "").replace(",", ".").replace("−", "-").strip()))
        if min(abs(v - a) for a in allowed) > tol:
            problems.append(f"unknown percentage: {p.strip()}")
    t = text.lower()
    says_up = any(w in t for w in UP_WORDS)
    says_down = any(w in t for w in DOWN_WORDS)
    g = metrics["monthly_growth_percent"]
    if g < 0 and says_up and not says_down:
        problems.append("says up, data says down")
    if g > 0 and says_down and not says_up:
        problems.append("says down, data says up")
    causes = [w for w in CAUSE_WORDS if w in t]
    return problems, causes

# Quick self-test of the verifier
_m = {"monthly_growth_percent": -5.56, "profit_margin_percent": 12.3}
for s in ["Sales declined by 5.56% last month.", "Sales fell 15% due to competition.", "Sales increased by 13.33%."]:
    print(f"{s!r:45} -> {verify_report(s, _m)}")

In [ ]:
results_b = []
if LLM_AVAILABLE:
    for i in truth:
        m = truth[i]
        text, sec = ask_llm([{"role": "system", "content": SYSTEM_B},
                             {"role": "user", "content": f"Metrics: {json.dumps(m)}"}])
        problems, causes = verify_report(text, m)
        results_b.append({"scenario": i, "seconds": round(sec, 2), "report": text,
                          "problems": "; ".join(problems), "blocked": bool(problems),
                          "invented_cause": bool(causes), "cause_words": ", ".join(causes)})
        print(f"scenario {i}: {sec:.1f}s | blocked={bool(problems)} | cause={bool(causes)}")
    res_b = pd.DataFrame(results_b)
    res_b.to_csv(f"{OUT}/exp3_hybrid.csv", index=False)
    display(res_b[["scenario", "seconds", "blocked", "problems", "invented_cause"]].head(10))
else:
    print("Skipped: no LLM available.")

## 7. Results summary and charts

In [ ]:
summary = {
    "model": MODEL, "n_scenarios": N_SCENARIOS, "n_months": N_MONTHS, "seed": SEED,
    "exp1_sort_wrong_value_%": round(pitfall["wrong_value"].mean() * 100, 1),
    "exp1_sort_wrong_direction_%": round(pitfall["wrong_direction"].mean() * 100, 1),
}
if LLM_AVAILABLE:
    summary.update({
        "A_valid_json_%": round(res_a["valid_json"].mean() * 100, 1),
        "A_total_correct_%": round(res_a["total_ok"].mean() * 100, 1),
        "A_growth_correct_%": round(res_a["growth_ok"].mean() * 100, 1),
        "A_direction_correct_%": round(res_a["direction_ok"].mean() * 100, 1),
        "A_median_total_error_%": round(res_a["total_rel_error_%"].median(), 3),
        "A_median_growth_error_pts": round(res_a["growth_abs_error_pts"].median(), 3),
        "A_mean_seconds": round(res_a["seconds"].mean(), 2),
        "B_numbers_correct_by_construction_%": 100.0,
        "B_reports_blocked_%": round(res_b["blocked"].mean() * 100, 1),
        "B_reports_with_cause_words_%": round(res_b["invented_cause"].mean() * 100, 1),
        "B_mean_seconds": round(res_b["seconds"].mean(), 2),
    })
with open(f"{OUT}/summary.json", "w") as f:
    json.dump(summary, f, indent=2)
pd.Series(summary).to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 2 if LLM_AVAILABLE else 1, figsize=(12 if LLM_AVAILABLE else 6, 4))
axes = np.atleast_1d(axes)

ax = axes[0]
vals = [summary["exp1_sort_wrong_value_%"], summary["exp1_sort_wrong_direction_%"]]
ax.bar(["Wrong growth value", "Wrong direction"], vals, color=["#4C72B0", "#C44E52"])
ax.set_ylim(0, 100); ax.set_ylabel("% of scenarios")
ax.set_title("Exp. 1: alphabetical month sort")
for k, v in enumerate(vals):
    ax.text(k, v + 2, f"{v:.0f}%", ha="center")

if LLM_AVAILABLE:
    ax = axes[1]
    labels = ["Total", "Growth", "Direction"]
    a = [summary["A_total_correct_%"], summary["A_growth_correct_%"], summary["A_direction_correct_%"]]
    b = [100, 100, 100 - summary["B_reports_blocked_%"]]
    x = np.arange(3)
    ax.bar(x - 0.2, a, 0.4, label="A: LLM computes", color="#C44E52")
    ax.bar(x + 0.2, b, 0.4, label="B: Python computes + check", color="#55A868")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 110); ax.set_ylabel("% correct")
    ax.set_title(f"Exp. 2 vs 3 ({MODEL}, {N_SCENARIOS} scenarios)")
    ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig(f"{OUT}/results.png", dpi=150)
plt.show()

## 7b. Figures to understand the results

Each figure is saved in `outputs/figures/` (PNG, 150 dpi) so it can be reused in an article or a README.
Grey = reference / truth, red = error, green = correct.

In [ ]:
FIG = f"{OUT}/figures"
os.makedirs(FIG, exist_ok=True)
GREY, RED, GREEN, BLUE = "#9AA0A6", "#C44E52", "#55A868", "#4C72B0"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def save(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

### Figure 1 — What the alphabetical sort really compares (company 0)
The correct code compares the last two months (Nov → Dec). The naive code, sorting month **names**,
compares Oct → Sep, two months that are not even the latest ones.

In [ ]:
g = data[data["scenario"] == 0].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g["month_name"], g["sales"], marker="o", color=GREY, label="Monthly sales")
for idx, col, lab in [((10, 11), GREEN, "Correct comparison: Nov → Dec"),
                      ((9, 8), RED, "Alphabetical sort compares: Oct → Sep")]:
    ax.plot(g["month_name"].iloc[list(idx)], g["sales"].iloc[list(idx)], marker="o", lw=3, color=col, label=lab)
true_g = truth[0]["monthly_growth_percent"]
naive_g = pitfall.loc[pitfall["scenario"] == 0, "naive_growth_%"].iloc[0]
ax.set_title(f"Company 0: true growth {true_g:+.2f}%  vs  naive code {naive_g:+.2f}%")
ax.set_ylabel("Sales")
ax.legend(frameon=False)
save(fig, "fig1_sort_pitfall_example")

### Figure 2 — Naive growth vs true growth (all companies)
Points on the dashed diagonal would be correct. Points in the red zones have the **wrong sign**:
the report would say "up" when sales went down, or the reverse.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
lim = max(pitfall["true_growth_%"].abs().max(), pitfall["naive_growth_%"].abs().max()) * 1.15
ax.fill_between([0, lim], -lim, 0, color=RED, alpha=0.08)
ax.fill_between([-lim, 0], 0, lim, color=RED, alpha=0.08)
ax.plot([-lim, lim], [-lim, lim], "--", color=GREY, lw=1)
ax.axhline(0, color=GREY, lw=0.8); ax.axvline(0, color=GREY, lw=0.8)
c = np.where(pitfall["wrong_direction"], RED, BLUE)
ax.scatter(pitfall["true_growth_%"], pitfall["naive_growth_%"], c=c, s=50, zorder=3)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel("True growth of last month (%)")
ax.set_ylabel("Growth from naive code (%)")
ax.set_title(f"Exp. 1: wrong sign in {pitfall['wrong_direction'].mean():.0%} of companies")
save(fig, "fig2_naive_vs_true_growth")

### Figure 3 — LLM growth vs true growth (Approach A)
Same reading as Figure 2, now for the numbers computed by the LLM.

In [ ]:
if LLM_AVAILABLE:
    ra = res_a.dropna(subset=["llm_growth"])
    fig, ax = plt.subplots(figsize=(6, 6))
    lim = max(ra["true_growth"].abs().max(), ra["llm_growth"].abs().max()) * 1.15
    ax.fill_between([0, lim], -lim, 0, color=RED, alpha=0.08)
    ax.fill_between([-lim, 0], 0, lim, color=RED, alpha=0.08)
    ax.plot([-lim, lim], [-lim, lim], "--", color=GREY, lw=1)
    ax.axhline(0, color=GREY, lw=0.8); ax.axvline(0, color=GREY, lw=0.8)
    c = np.where(ra["growth_ok"], GREEN, np.where(ra["direction_ok"], BLUE, RED))
    ax.scatter(ra["true_growth"], ra["llm_growth"], c=c, s=50, zorder=3)
    for k, (_, r) in enumerate(ra[~ra["direction_ok"]].iterrows()):
        ax.annotate(f"company {int(r.scenario)}", (r.true_growth, r.llm_growth),
                    textcoords="offset points", xytext=(6, 8 if k % 2 == 0 else -14), fontsize=9, color=RED)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel("True growth (%)"); ax.set_ylabel("Growth computed by the LLM (%)")
    ax.set_title("Exp. 2: green = exact, blue = right sign only, red = wrong sign")
    save(fig, "fig3_llm_vs_true_growth")

### Figure 4 — Size of the LLM's errors, company by company (Approach A)
Left: error on total sales (log scale, because one error is huge). Right: error on growth.
The dashed line is the tolerance: bars to the right of it count as wrong.

In [ ]:
if LLM_AVAILABLE:
    ra = res_a.sort_values("scenario")
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    labels = [f"company {int(s)}" for s in ra["scenario"]]
    tot = ra["total_rel_error_%"].clip(lower=1e-3)
    axes[0].barh(labels, tot, color=np.where(ra["total_ok"], GREEN, RED))
    axes[0].axvline(TOLERANCE_REL * 100, ls="--", color=GREY)
    axes[0].set_xscale("log"); axes[0].invert_yaxis()
    axes[0].set_xlabel("Relative error on total sales (%, log scale)")
    axes[0].set_title(f"Total sales: {ra['total_ok'].mean():.0%} within {TOLERANCE_REL*100:.1f}%")
    axes[1].barh(labels, ra["growth_abs_error_pts"], color=np.where(ra["growth_ok"], GREEN, RED))
    axes[1].axvline(TOLERANCE_PCT, ls="--", color=GREY)
    axes[1].set_xlabel("Absolute error on growth (percentage points)")
    axes[1].set_title(f"Growth: {ra['growth_ok'].mean():.0%} within {TOLERANCE_PCT} pt")
    save(fig, "fig4_llm_errors_per_company")

### Figure 5 — Direction: what the LLM said vs the truth (Approach A)
A confusion table: the diagonal is correct, the other cells are reports with the wrong direction.

In [ ]:
if LLM_AVAILABLE:
    cm = pd.crosstab(res_a["true_direction"], res_a["llm_direction"]).reindex(
        index=["up", "down"], columns=["up", "down"], fill_value=0)
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm.values, cmap="Greys", vmin=0, vmax=cm.values.max() * 4)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm.values[i, j], ha="center", va="center", fontsize=16,
                    color=GREEN if i == j else RED, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["says up", "says down"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["truly up", "truly down"])
    ax.set_title("Exp. 2: direction announced by the LLM")
    ax.spines[:].set_visible(False)
    save(fig, "fig5_direction_confusion")

### Figure 6 — Time per answer on this machine
Each dot is one company. Approach B reads six numbers instead of a table, so it is usually faster.

In [ ]:
if LLM_AVAILABLE:
    fig, ax = plt.subplots(figsize=(7, 4))
    for k, (df_, name, col) in enumerate([(res_a, "A: LLM computes", RED), (res_b, "B: Python computes + LLM explains", GREEN)]):
        y = np.full(len(df_), k) + rng.uniform(-0.12, 0.12, len(df_))
        ax.scatter(df_["seconds"], y, color=col, alpha=0.7, s=40)
        ax.plot([df_["seconds"].median()] * 2, [k - 0.25, k + 0.25], color="black", lw=2)
        ax.text(df_["seconds"].median(), k + 0.32, f"median {df_['seconds'].median():.1f} s", ha="center", fontsize=9)
    ax.set_yticks([0, 1]); ax.set_yticklabels(["A: LLM computes", "B: Python + LLM"])
    ax.set_ylim(-0.6, 1.7)
    ax.set_xlabel("Seconds per answer")
    device = "GPU" if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0 else "CPU"
    ax.set_title(f"Latency of {MODEL} on {device}")
    save(fig, "fig6_latency")

### Figure 7 — What the verifier found in the Approach B reports

In [ ]:
if LLM_AVAILABLE:
    counts = pd.Series({
        "Accepted, no issue": int((~res_b["blocked"] & ~res_b["invented_cause"]).sum()),
        "Accepted, but cause word": int((~res_b["blocked"] & res_b["invented_cause"]).sum()),
        "Blocked: wrong number or direction": int(res_b["blocked"].sum()),
    })
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(counts.index, counts.values, color=[GREEN, "#DD8452", RED])
    for i, v in enumerate(counts.values):
        ax.text(v + 0.2, i, str(v), va="center")
    ax.invert_yaxis(); ax.set_xlim(0, len(res_b) * 1.1)
    ax.set_xlabel(f"Number of reports (out of {len(res_b)})")
    ax.set_title("Exp. 3: verification of the LLM-written reports")
    save(fig, "fig7_verifier_outcomes")

### Figure 8 — How the hybrid pipeline works

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.4))
ax.axis("off")
steps = [("Raw sales\n(CSV)", GREY), ("Python + Pandas\ncompute metrics", BLUE), ("LLM writes\nthe explanation", "#8172B3"),
         ("Verifier checks\n% , direction, causes", "#DD8452"), ("Report to\nmanager", GREEN)]
for i, (txt, col) in enumerate(steps):
    x = i * 2.2
    ax.add_patch(plt.Rectangle((x, 0), 1.8, 1, color=col, alpha=0.18, ec=col, lw=1.5))
    ax.text(x + 0.9, 0.5, txt, ha="center", va="center", fontsize=11)
    if i < len(steps) - 1:
        ax.annotate("", xy=(x + 2.2, 0.5), xytext=(x + 1.8, 0.5), arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("blocked → back to review", xy=(3 * 2.2 + 0.9, 0), xytext=(3 * 2.2 + 0.9, -0.55),
            ha="center", fontsize=10, color=RED, arrowprops=dict(arrowstyle="<-", color=RED))
ax.set_xlim(-0.2, 11); ax.set_ylim(-0.8, 1.2)
save(fig, "fig8_pipeline")
print("Figures saved:", sorted(os.listdir(FIG)))

## 8. Copy this output back to the conversation

The cell below prints everything needed to write the article: the summary,
a few example reports, and the environment.

In [ ]:
print("=== SUMMARY ===")
print(json.dumps(summary, indent=2))
print("\n=== ENVIRONMENT ===")
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "|", platform.platform())
if LLM_AVAILABLE:
    print("\n=== APPROACH A: 5 WORST GROWTH ERRORS ===")
    print(res_a.sort_values("growth_abs_error_pts", ascending=False)
              [["scenario", "llm_growth", "true_growth", "llm_direction", "true_direction"]].head(5).to_string(index=False))
    print("\n=== APPROACH B: BLOCKED REPORTS (up to 3) ===")
    for _, r in res_b[res_b["blocked"]].head(3).iterrows():
        print(f"- scenario {r.scenario}: {r.problems}\n  {r.report[:400]}\n")
    print("=== APPROACH B: ONE ACCEPTED REPORT ===")
    ok = res_b[~res_b["blocked"]]
    if len(ok):
        print(ok.iloc[0]["report"])
print("\nFiles saved in outputs/:", sorted(os.listdir(OUT)))

## Limits of this experiment

- Synthetic data: real data has more noise, missing values and odd formats.
- One model, temperature 0: results may differ with other models or settings.
- The verifier checks percentages and direction only; it does not prove a report is fully correct.
- Cause detection uses a keyword list: it is a rough signal, not a measurement.